# Instrument and Verify: Add Tracing to an Existing App and Prove It Worked

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/instrument-and-verify). Run the cells in order in a fresh Python 3.12 kernel.

## Install

This notebook runs both tracing tracks in separate processes. Use a fresh workspace so the local captures and downloaded example do not overwrite an existing project.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

workspace = Path(tempfile.mkdtemp(prefix="traced-support-"))
os.chdir(workspace)
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "fi-instrumentation-otel==1.5.1", "traceai-openai==0.1.16", "openai==3.57.0"], check=True);

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "opentelemetry-sdk", "opentelemetry-exporter-otlp-proto-http", "openai==3.57.0"], check=True);

### Download the checker

Check its digest before running it. The local capture contains prompt and response content; use synthetic inputs only.

In [ ]:
%%bash
set -eu
mkdir -p observability/futureagi
curl -fsSL https://docs.futureagi.com/fi_verify.py -o observability/futureagi/fi_verify.py
python -c 'import hashlib, pathlib; assert hashlib.sha256(pathlib.Path("observability/futureagi/fi_verify.py").read_bytes()).hexdigest() == "75ce8d9cb5bafa1cbeea42affe8b464433ab97e438a53410f27678855d484de7", "unexpected checker digest"'
printf '\n.fi_verify/\n' >> .gitignore

## Set your credentials

Set the named environment variables or Colab secrets before running, or enter them at the hidden prompts. Values are not displayed. For a self-hosted deployment, set `FI_BASE_URL` to your API and `FI_TRACES_URL` to your collector before continuing.

In [ ]:
import os
from getpass import getpass


def require_secret(name):
    if os.environ.get(name):
        return
    value = None
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass
    if not value:
        value = getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value


for name in ('FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY'):
    require_secret(name)

os.environ.setdefault("FI_TRACES_URL", "https://api.futureagi.com")
from uuid import uuid4
os.environ["FI_PROJECT_NAME"] = "ridgeline-traced-support"
os.environ["FI_ENDPOINT"] = os.environ["FI_TRACES_URL"].rstrip("/") + "/tracer/v1/traces"
os.environ["FI_VERIFY"] = "1"
os.environ["FI_SESSION_ID"] = f"support-{uuid4().hex}"

## Prove the keys before writing code that depends on them

In [ ]:
%%bash
set -eu
python observability/futureagi/fi_verify.py preflight

## Make one request arrive as one tree

Save the setup module for each tracing track.

In [ ]:
%%writefile observability/futureagi/setup_sdk.py
import os

from fi_instrumentation import FITracer, register, using_attributes
from fi_instrumentation.fi_types import ProjectType
from . import fi_verify

provider = register(project_name=os.environ["FI_PROJECT_NAME"],
                    project_type=ProjectType.OBSERVE, verbose=False)
tracer = FITracer(provider.get_tracer("support"))


def request_scope(session_id, user_id):
    return using_attributes(session_id=session_id, user_id=user_id)


if os.getenv("FI_VERIFY") == "1":
    fi_verify.attach(provider)

In [ ]:
%%writefile observability/futureagi/setup_otel.py
import os
from contextlib import contextmanager
from contextvars import ContextVar

from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import SpanProcessor, TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from . import fi_verify

provider = TracerProvider(resource=Resource.create({
    "project_name": os.environ["FI_PROJECT_NAME"], "project_type": "observe",
}))
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(
    endpoint=os.environ["FI_ENDPOINT"],
    headers={"X-Api-Key": os.environ["FI_API_KEY"],
             "X-Secret-Key": os.environ["FI_SECRET_KEY"]},
)))
_scope = ContextVar("support_scope", default={})


class ScopeProcessor(SpanProcessor):
    def on_start(self, span, parent_context=None):
        span.set_attributes(_scope.get())


@contextmanager
def request_scope(session_id, user_id):
    token = _scope.set({"session.id": session_id, "user.id": user_id})
    try:
        yield
    finally:
        _scope.reset(token)


provider.add_span_processor(ScopeProcessor())
tracer = provider.get_tracer("support")
if os.getenv("FI_VERIFY") == "1":
    fi_verify.attach(provider)

## Carry session and user from the edge

In [ ]:
%%writefile observability/futureagi/request.py
from contextlib import contextmanager


@contextmanager
def support_turn(setup, question, session_id, user_id):
    with setup.request_scope(session_id, user_id):
        with setup.tracer.start_as_current_span("support.turn", attributes={
            "gen_ai.span.kind": "CHAIN", "input.value": question,
        }) as root:
            yield root

## Write the model, messages and usage where they belong

In [ ]:
%%writefile support.py
import os
import sys
from contextlib import nullcontext

from openai import OpenAI
from observability.futureagi.request import support_turn

if sys.argv[1:] == ["sdk"]:
    from observability.futureagi import setup_sdk as setup
    from traceai_openai import OpenAIInstrumentor
    OpenAIInstrumentor().instrument(tracer_provider=setup.provider)
    manual = False
elif sys.argv[1:] == ["otel"]:
    from observability.futureagi import setup_otel as setup
    manual = True
else:
    raise SystemExit("Choose a tracing track: sdk or otel")

question = "Can I return an unused tent after two weeks?"
model = "gpt-4o-mini"
with support_turn(setup, question, os.environ["FI_SESSION_ID"], "shopper-demo") as root:
    with setup.tracer.start_as_current_span("policy.lookup", attributes={
        "gen_ai.span.kind": "TOOL", "input.value": "tent returns",
    }) as tool:
        policy = {"tent returns": "Unused tents can be returned within 30 days."}["tent returns"]
        tool.set_attribute("output.value", policy)
    context = setup.tracer.start_as_current_span("support.reply", attributes={
        "gen_ai.span.kind": "LLM", "gen_ai.request.model": model,
        "gen_ai.provider.name": "openai", "input.value": question,
    }) if manual else nullcontext()
    with context as span, OpenAI() as client:
        response = client.chat.completions.create(model=model, messages=[
            {"role": "system", "content": "Answer in one sentence using this policy: " + policy},
            {"role": "user", "content": question},
        ])
        answer = response.choices[0].message.content
        if not answer:
            raise ValueError("Expected a text reply")
        if manual:
            usage = response.usage
            span.set_attributes({"output.value": answer,
                                 "gen_ai.usage.input_tokens": usage.prompt_tokens,
                                 "gen_ai.usage.output_tokens": usage.completion_tokens,
                                 "gen_ai.usage.total_tokens": usage.total_tokens})
    root.set_attribute("output.value", answer)
    trace_id = f"{root.get_span_context().trace_id:032x}"
setup.provider.force_flush()
print(answer)
print(f"Trace ID: {trace_id}")

## Run the ten gates

Run the SDK track and check it before starting the OpenTelemetry track. Each run replaces the local capture. Note each trace ID so you can find the stored span tree in Observe.

In [ ]:
%%bash
set -eu
python support.py sdk
python observability/futureagi/fi_verify.py check

In [ ]:
%%bash
set -eu
python support.py otel
python observability/futureagi/fi_verify.py check

## The same steps on an existing repository

Install the matching OpenAI Agents and instrumentor versions and obtain the corresponding public customer-service example.

In [ ]:
%%bash
set -eu
python -m pip install openai-agents==0.25.0 traceai-openai-agents==0.1.8
git clone --depth 1 --branch v0.25.0 https://github.com/openai/openai-agents-python.git openai-agents-python

In [ ]:
%%writefile run_traced.py
import asyncio
import os

from agents import RunConfig, Runner, set_default_openai_api
from traceai_openai_agents import OpenAIAgentsInstrumentor
from examples.customer_service.main import AirlineAgentContext, triage_agent
from observability.futureagi import setup_sdk as setup
from observability.futureagi.request import support_turn


async def main():
    OpenAIAgentsInstrumentor().instrument(tracer_provider=setup.provider)
    set_default_openai_api("chat_completions")
    question = "What is the baggage allowance for my flight?"
    with support_turn(setup, question, os.environ["FI_SESSION_ID"], "shopper-demo") as root:
        result = await Runner.run(triage_agent, question, context=AirlineAgentContext(),
                                  run_config=RunConfig(model="gpt-4o-mini"))
        answer = str(result.final_output)
        root.set_attribute("output.value", answer)
        trace_id = f"{root.get_span_context().trace_id:032x}"
    setup.provider.force_flush()
    print(answer)
    print(f"Trace ID: {trace_id}")


if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
%%bash
set -eu
PYTHONPATH="$PWD/openai-agents-python" python run_traced.py
python observability/futureagi/fi_verify.py check

Look for `GREEN LIGHT achieved` on all three checks. In Observe, compare the three trace IDs in `ridgeline-traced-support`: each should contain its own root and model spans and share the same session.